In [1]:
#INSTALLATION
# Exécute cette cellule une seule fois
!pip install nltk spacy
!python -m spacy download fr_core_news_sm


[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
      --------------------------------------- 0.3/16.3 MB ? eta -:--:--
     --- ------------------------------------ 1.3/16.3 MB 5.6 MB/s eta 0:00:03
     ------- -------------------------------- 2.9/16.3 MB 6.2 MB/s eta 0:00:03
     ---------- ----------------------------- 4.5/16.3 MB 6.5 MB/s eta 0:00:02
     -------------- ------------------------- 5.8/16.3 MB 6.5 MB/s eta 0:00:02
     ------------------ --------------------- 7.3/16.3 MB 6.8 MB/s eta 0:00:02
     --------------------- ------------------ 8.9/16.3 MB 7.0 MB/s eta 0:00:02
     -------------------------- ------------- 10.7/16.3 MB 7.2 MB/s eta 0:00:01
     ------------------------------ --------- 12.3/16.3 MB 7.3 MB/s eta 0:00:01
     ----------------------------------- ---- 14.4/16.3 MB 7.5 MB/s eta 0:00:01
     ---------------------------------------  16.0/16.3 MB 7.6 MB/s eta 0:00:01
     ---------------------------------------- 16.3/16.3 MB 7.4


[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#IMPORTS 
import re
import nltk
import spacy

nltk.download('punkt')
nltk.download('punkt_tab')

nlp_spacy = spacy.load("fr_core_news_sm")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\THUNDERROBOT\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\THUNDERROBOT\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


In [11]:
phrases = [
    "J'adore ce film !!! 😍 #cinéma",
    "I don't like it...",
    "Quelle belle journée ! #sun",
    "It's a great day!!!",
    "Je n'aime pas ça...",
]

In [7]:
def split_contractions(tokens):
    # Liste des contractions anglaises courantes
    contractions = {
        "don't": ["do", "n't"],
        "doesn't": ["does", "n't"],
        "didn't": ["did", "n't"],
        "won't": ["will", "n't"],
        "can't": ["ca", "n't"],
        "isn't": ["is", "n't"],
        "aren't": ["are", "n't"],
        "i'm": ["i", "'m"],
        "it's": ["it", "'s"],
        "that's": ["that", "'s"],
        "what's": ["what", "'s"],
    }
    new_tokens = []
    for token in tokens:
        if token in contractions:
            new_tokens.extend(contractions[token])
        else:
            new_tokens.append(token)
    return new_tokens

In [8]:
def my_tokenizer(text):
     # Pattern : hashtag OU mot avec apostrophe possible OU ponctuation isolée
    pattern = r"#\w+|\w+(?:'\w+)?|[^\w\s]"

    # Tout mettre en minuscules
    text = text.lower()

     # Trouver tous les tokens
    tokens = re.findall(pattern, text)
    tokens = split_contractions(tokens)
    
    return tokens

In [12]:
import spacy

# Charger le modèle français
nlp = spacy.load("fr_core_news_sm")

def spacy_tokenizer(text):
    doc = nlp(text)
    return [token.text for token in doc]

In [13]:
print(my_tokenizer("J'adore ce film !!!"))

["j'adore", 'ce', 'film', '!', '!', '!']


In [14]:
# Test pour chaque phrase
for phrase in phrases:
    print("Phrase:", phrase)
    print("Mon tokenizer :", my_tokenizer(phrase))
    print("NLTK          :", nltk.word_tokenize(phrase))
    print("spaCy         :", spacy_tokenizer(phrase))
    print("-" * 60)

Phrase: J'adore ce film !!! 😍 #cinéma
Mon tokenizer : ["j'adore", 'ce', 'film', '!', '!', '!', '😍', '#cinéma']
NLTK          : ["J'adore", 'ce', 'film', '!', '!', '!', '😍', '#', 'cinéma']
spaCy         : ["J'", 'adore', 'ce', 'film', '!', '!', '!', '😍', '#', 'cinéma']
------------------------------------------------------------
Phrase: I don't like it...
Mon tokenizer : ['i', 'do', "n't", 'like', 'it', '.', '.', '.']
NLTK          : ['I', 'do', "n't", 'like', 'it', '...']
spaCy         : ['I', "don'", 't', 'like', 'it', '...']
------------------------------------------------------------
Phrase: Quelle belle journée ! #sun
Mon tokenizer : ['quelle', 'belle', 'journée', '!', '#sun']
NLTK          : ['Quelle', 'belle', 'journée', '!', '#', 'sun']
spaCy         : ['Quelle', 'belle', 'journée', '!', '#', 'sun']
------------------------------------------------------------
Phrase: It's a great day!!!
Mon tokenizer : ['it', "'s", 'a', 'great', 'day', '!', '!', '!']
NLTK          : ['It', "'s",

## 📋 RÉSUMÉ COMPLET TP 1 — TOKENIZER MAISON

```markdown
# 📋 RÉSUMÉ TP 1 — Tokenizer Maison

---

## 1. LES IMPORTS ET LEUR RÔLE

| Import | À quoi ça sert |
|--------|----------------|
| `re` | Créer des patterns regex pour découper le texte |
| `nltk` | Tokenizer pré-entraîné (word_tokenize) — référence académique |
| `spacy` | Tokenizer industriel rapide — utilisé en production |

| Ligne | Rôle |
|-------|------|
| `nltk.download('punkt')` | Télécharge les règles de tokenization NLTK |
| `nltk.download('punkt_tab')` | Corrige un bug de dépendance NLTK |
| `spacy.load("fr_core_news_sm")` | Charge un modèle français spaCy (tokenizer + POS + lemmatizer) |

---

## 2. LA MENTALITÉ

> **Il n'existe pas de "bon" tokenizer universel.**  
> Tout dépend de la tâche finale :  
> - Analyse de sentiment → garder `!!!` et `😍` (signaux d'émotion)  
> - Topic modeling → les virer (bruit)  
> - NER (entités nommées) → garder la casse (`New York`)  
> - Chatbot → garder la ponctuation (`?` = question)

---

## 3. LA REGEX — MÉTHODE DE CONSTRUCTION

| Étape | Action |
|-------|--------|
| 1 | Décrire en français ce qu'on veut capturer |
| 2 | Remplacer chaque morceau par son symbole regex |
| 3 | Tester sur un exemple |
| 4 | Ajuster |

### Symboles clés

| Symbole | Signification | Exemple |
|---------|---------------|---------|
| `\w` | Lettre, chiffre, underscore | `a`, `B`, `9` |
| `\W` | Tout SAUF `\w` | `!`, `#`, `@` |
| `\s` | Espace, tabulation | ` ` |
| `+` | 1 ou plusieurs | `\w+` → `hello` |
| `?` | 0 ou 1 (optionnel) | `\w?` → `a` ou rien |
| `()` | Groupe capturant | `(?:...)` = non-capturant |
| `[]` | Un caractère parmi | `[abc]` → `a`, `b`, ou `c` |
| `|` | OU logique | `a|b` → `a` ou `b` |

### Notre pattern final

```python
r"#\w+|\w+(?:'\w+)?|[^\w\s]"
```

| Partie | Capture |
|--------|---------|
| `#\w+` | Hashtags (`#cinéma`) |
| `\w+(?:'\w+)?` | Mots avec apostrophe optionnelle (`don't`, `j'adore`) |
| `[^\w\s]` | Ponctuation et symboles isolés (`!`, `😍`) |
| `|` | OU entre chaque règle |

### Gestion des contractions anglaises

```python
def split_contractions(tokens):
    contractions = {
        "don't": ["do", "n't"],
        "doesn't": ["does", "n't"],
        "isn't": ["is", "n't"],
        "won't": ["will", "n't"],
        "can't": ["ca", "n't"],
        "i'm": ["i", "'m"],
        "it's": ["it", "'s"],
    }
    new_tokens = []
    for token in tokens:
        if token in contractions:
            new_tokens.extend(contractions[token])
        else:
            new_tokens.append(token)
    return new_tokens
```

---

## 4. DIVERGENCES ENTRE LES 3 TOKENIZERS

### Cas 1 : Hashtag `#cinéma`

| Tokenizer | Résultat |
|-----------|----------|
| Mon tokenizer | `['#cinéma']` |
| NLTK | `['#', 'cinéma']` |
| spaCy | `['#', 'cinéma']` |

**Pourquoi ?** Ma regex `#\w+` capture le hashtag entier (unité de sens). NLTK et spaCy traitent `#` comme ponctuation.

**✅ Vainqueur :** Mon tokenizer (garde l'information du hashtag).

---

### Cas 2 : Contraction française `n'aime`

| Tokenizer | Résultat |
|-----------|----------|
| Mon tokenizer | `["n'aime"]` |
| NLTK | `["n'aime"]` |
| spaCy | `["n'", 'aime']` |

**Pourquoi ?** spaCy (modèle français) reconnaît la négation `n'` = `ne`. Mon tokenizer et NLTK gardent la forme entière.

**✅ Vainqueur :** spaCy (meilleur pour le français).

---

### Cas 3 : Contraction anglaise `don't`

| Tokenizer | Résultat |
|-----------|----------|
| Mon tokenizer | `['do', "n't"]` |
| NLTK | `['do', "n't"]` |
| spaCy | `["don'", 't']` |

**Pourquoi ?** Mon tokenizer (avec split_contractions) et NLTK coupent correctement. spaCy (modèle français) ne connaît pas l'anglais.

**✅ Vainqueur :** Mon tokenizer et NLTK.

---

## 5. CE QUE J'AI APPRIS

1. Un tokenizer découpe du texte en mots/symboles
2. Les regex sont l'outil de base pour créer son tokenizer
3. NLTK, spaCy et un tokenizer maison font des choix différents
4. Aucun tokenizer n'est universel — tout dépend de la langue et de la tâche
5. 90% des bugs NLP viennent d'un mauvais prétraitement
6. Les contractions doivent être traitées selon la langue cible
```